In [1]:
import os
import cv2
import random
import pickle
import math
import numpy as np
from tqdm import tqdm
import mediapipe as mp
import matplotlib.pyplot as plt
from scipy.signal import medfilt
from collections import defaultdict
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

2025-04-13 23:38:54.478671: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-13 23:38:54.487537: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744605534.496750    1462 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744605534.499600    1462 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-13 23:38:54.511399: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
class Config():
    def __init__(self):        
        self.frame_l = 64 # the length of frames
        self.joint_n = 33 # the number of joints
        self.joint_d = 3 # the dimension of classes        
        self.save_dir = r"/home/nyam/Ms_Thesis/Autism/Project_v3/Code_Base/DataProcessing/v2/skel_clean_aug"
        self.data_dir = r"/home/nyam/Ms_Thesis/Autism/Project_v3/Code_Base/DataProcessing/v2/skel_clean"
        self.dataset_info = r"/home/nyam/Ms_Thesis/Autism/Project_v3/Code_Base/DataProcessing/v2/dataset_info_clean.txt"
        self.dataset_info_aug = r"/home/nyam/Ms_Thesis/Autism/Project_v3/Code_Base/DataProcessing/v2/dataset_info_clean_aug.txt"

C = Config()

In [3]:
# Function to read dataset info from file
def read_dataset_info(file_path):
    dataset_info = []
    with open(file_path, "r") as info_file:
        for line in info_file:
            # Strip any leading or trailing whitespace and split the line by tabs
            info = line.strip().split("\t")
            if info[1] != "HandAction":
                dataset_info.append(info)
    return dataset_info

augumentation functions

In [4]:
# Define augmentation functions
def translate_skeleton(skeleton, max_translation=0.05):
    """Translate skeleton by a random amount."""
    translation = np.random.uniform(-max_translation, max_translation, size=(1, skeleton.shape[1], skeleton.shape[2]))
    return skeleton + translation

def scale_skeleton(skeleton, scale_range=(0.8, 1.2)):
    """Scale skeleton by a random factor."""
    scale_factor = np.random.uniform(*scale_range, size=(1, 1, skeleton.shape[2]))
    return skeleton * scale_factor

def rotate_skeleton(skeleton, max_angle=15):
    """Rotate skeleton by a random angle around the Z-axis."""
    angle = np.radians(np.random.uniform(-max_angle, max_angle))
    rotation_matrix = np.array([
        [np.cos(angle), -np.sin(angle), 0],
        [np.sin(angle), np.cos(angle), 0],
        [0, 0, 1]
    ])
    return np.dot(skeleton, rotation_matrix)

def apply_augmentations(skeleton, num_augments=5):
    augmented_skeletons = []

    if skeleton.size == 0:
        print("Warning: Empty skeleton array; no augmentations applied.")
        return augmented_skeletons

    #print("Original skeleton shape:", skeleton.shape)

    for _ in range(num_augments):
        augmented_skeleton = skeleton.copy()
        augmented_skeleton = translate_skeleton(augmented_skeleton)
        augmented_skeleton = scale_skeleton(augmented_skeleton)
        augmented_skeleton = rotate_skeleton(augmented_skeleton)
        augmented_skeletons.append(augmented_skeleton)

    return augmented_skeletons

*Compute augumentations needed to balance dataset*

In [11]:
def compute_augmentation_needs(file_path):
    """
    Reads dataset_info.txt and computes the number of augmentations needed per subject
    to balance both per-subject clips (min 7) and total class clips.

    Returns:
        dict: A dictionary with subject_id as keys and number of augmentations per clip as values.
    """
    # Dictionary to store subject clip counts and class labels
    subject_clip_count = defaultdict(int)
    subject_class_mapping = {}

    # Read dataset and count clips per subject
    with open(file_path, 'r') as file:
        for line in file:
            parts = line.strip().split('\t')
            class_label = parts[1]  # Action class
            if class_label == "HandAction":
                continue  # Skip "HandAction" class

            filename = parts[2]  # Filename containing subject ID

            # Extract subject ID from filename (e.g., "Spinning_18" from "v_Spinning_18_action_seg_slice_6.npy")
            file_parts = filename.split('_')
            subject_id = f"{file_parts[1]}_{file_parts[2]}"

            # Count clips per subject
            subject_clip_count[subject_id] += 1

            # Map subjects to their class labels
            subject_class_mapping[subject_id] = class_label

    # Compute total clips per class before augmentation
    class_clip_counts_before = defaultdict(int)
    for subject, count in subject_clip_count.items():
        class_label = subject_class_mapping[subject]
        class_clip_counts_before[class_label] += count

    # Print number of clips per class before augmentation
    print("Number of clips per class BEFORE augmentation:")
    for class_label, count in class_clip_counts_before.items():
        print(f"{class_label}: {count}")

    TARGET_CLIPS_PER_SUBJECT = 5

    # Dictionary to store augmentation needs per subject
    subject_augmentation_needs = {}

    # Compute augmentation needs per subject
    for subject in subject_clip_count:
        current_clips = subject_clip_count[subject]
        missing_clips = max(0, TARGET_CLIPS_PER_SUBJECT - current_clips)
        subject_augmentation_needs[subject] = missing_clips

    # Compute total clips per class AFTER augmentation
    class_clip_counts_after = defaultdict(int)
    for subject, count in subject_clip_count.items():
        class_label = subject_class_mapping[subject]
        class_clip_counts_after[class_label] += count + subject_augmentation_needs[subject]

    # Print number of clips per class after augmentation
    print("\nNumber of clips per class AFTER augmentation:")
    for class_label, count in class_clip_counts_after.items():
        print(f"{class_label}: {count}")

    return subject_augmentation_needs, class_clip_counts_before, class_clip_counts_after

# Example usage:
file_path = C.dataset_info  # Replace with actual dataset path
augmentation_needs, clips_before, clips_after = compute_augmentation_needs(file_path)

# Print the computed augmentation dictionary
print("\nAugmentation needs per subject:")
print(augmentation_needs)


Number of clips per class BEFORE augmentation:
Spinning: 92
HeadBanging: 87
ArmFlapping: 80

Number of clips per class AFTER augmentation:
Spinning: 160
HeadBanging: 150
ArmFlapping: 150

Augmentation needs per subject:
{'Spinning_54': 4, 'HeadBanging_10': 3, 'ArmFlapping_10': 0, 'ArmFlapping_24': 4, 'HeadBanging_05': 2, 'ArmFlapping_15': 3, 'ArmFlapping_09': 1, 'ArmFlapping_37': 4, 'HeadBanging_39': 4, 'ArmFlapping_45': 0, 'HeadBanging_28': 1, 'Spinning_14': 3, 'Spinning_63': 1, 'HeadBanging_19': 2, 'ArmFlapping_34': 2, 'ArmFlapping_40': 1, 'HeadBanging_17': 0, 'ArmFlapping_44': 0, 'HeadBanging_53': 1, 'HeadBanging_48': 0, 'Spinning_21': 2, 'Spinning_44': 0, 'ArmFlapping_49': 3, 'HeadBanging_49': 0, 'Spinning_22': 4, 'Spinning_18': 0, 'Spinning_37': 1, 'ArmFlapping_51': 3, 'Spinning_30': 1, 'HeadBanging_52': 0, 'HeadBanging_27': 2, 'HeadBanging_06': 2, 'HeadBanging_33': 1, 'Spinning_19': 3, 'ArmFlapping_16': 0, 'Spinning_28': 2, 'ArmFlapping_36': 0, 'Spinning_46': 3, 'HeadBanging_38':

In [12]:
def Augu_video_clips_subjectwise(data_info, augmentation_dict):
    """
    Processes each subject individually:
    - Saves original skeletons
    - Applies required number of augmentations
    - Distributes augmentations evenly across that subject's clips
    - Writes all info to the dataset info file
    """
    sno = 1  # Serial number tracker

    # Group clips by subject_id
    subject_clips = defaultdict(list)
    for serial_number, action_name, skel in data_info:
        parts = skel.split('_')
        subject_id = f"{parts[1]}_{parts[2]}"
        subject_clips[subject_id].append((action_name, skel))

    for subject_id, clips in subject_clips.items():
        # Save original clips
        for action_name, skel in clips:
            skeleton_path = os.path.join(C.data_dir, skel)
            try:
                skeleton = np.load(skeleton_path).astype('float32')
                np.save(os.path.join(C.save_dir, skel), skeleton)

                with open(C.dataset_info_aug, "a") as info_file:
                    info_file.write(f"{sno}\t{action_name}\t{skel}\n")
                    sno += 1

            except Exception as e:
                print(f"Error saving original {skel}: {e}")
                continue
                
        total_augs = augmentation_dict.get(subject_id, 0)
        if total_augs <= 0:
            continue  # Skip if no augmentation needed

        print(f"\nProcessing Subject: {subject_id} | Clips: {len(clips)} | Augs needed: {total_augs}")


        # Distribute augmentations across clips as evenly as possible
        clip_count = len(clips)
        aug_counts = [total_augs // clip_count] * clip_count
        for i in range(total_augs % clip_count):
            aug_counts[i] += 1  # Distribute remainder

        # Apply augmentations
        for idx, (action_name, skel) in enumerate(clips):
            skeleton_path = os.path.join(C.data_dir, skel)
            try:
                skeleton = np.load(skeleton_path).astype('float32')
                n_aug = aug_counts[idx]

                if n_aug == 0:
                    continue

                augmented_versions = apply_augmentations(skeleton, n_aug)

                for j, aug_skel in enumerate(augmented_versions):
                    aug_filename = f"v_{subject_id}_aug_{idx}_{j}.npy"
                    np.save(os.path.join(C.save_dir, aug_filename), aug_skel)

                    with open(C.dataset_info_aug, "a") as info_file:
                        info_file.write(f"{sno}\t{action_name}\t{aug_filename}\n")
                        sno += 1

            except Exception as e:
                print(f"Error augmenting {skel}: {e}")
                continue

        print(f"Completed subject {subject_id} | Total augmentations done: {total_augs}")

    print("\nAll subjects processed successfully.")

if __name__ == "__main__":      
    data_info = read_dataset_info(C.dataset_info)
    Augu_video_clips_subjectwise(data_info, augmentation_needs)


Processing Subject: Spinning_54 | Clips: 1 | Augs needed: 4
Completed subject Spinning_54 | Total augmentations done: 4

Processing Subject: HeadBanging_10 | Clips: 2 | Augs needed: 3
Completed subject HeadBanging_10 | Total augmentations done: 3

Processing Subject: ArmFlapping_24 | Clips: 1 | Augs needed: 4
Completed subject ArmFlapping_24 | Total augmentations done: 4

Processing Subject: HeadBanging_05 | Clips: 3 | Augs needed: 2
Completed subject HeadBanging_05 | Total augmentations done: 2

Processing Subject: ArmFlapping_15 | Clips: 2 | Augs needed: 3
Completed subject ArmFlapping_15 | Total augmentations done: 3

Processing Subject: ArmFlapping_09 | Clips: 4 | Augs needed: 1
Completed subject ArmFlapping_09 | Total augmentations done: 1

Processing Subject: ArmFlapping_37 | Clips: 1 | Augs needed: 4
Completed subject ArmFlapping_37 | Total augmentations done: 4

Processing Subject: HeadBanging_39 | Clips: 1 | Augs needed: 4
Completed subject HeadBanging_39 | Total augmentation